# Training

In [ ]:
!pip install optuna catboost

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import optuna
from tqdm.auto import tqdm
from sklearn.linear_model import ElasticNet
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Configuration ---
DATA_DIR = '/content/drive/MyDrive/POLALIGNLLM/Datasets'
MODEL_DIR = '/content/drive/MyDrive/POLALIGNLLM/Models'
TARGETS = ['lrgen', 'lrecon', 'galtan']

datasets = {
    '09': 'd_09.csv',
    '14': 'd_14.csv',
    '19': 'd_19.csv'
}

# Ensure the model directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

# Suppress Optuna's default logging to keep the notebook clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Execution Loop ---
for year, file_name in datasets.items():
    print(f"\n{'='*55}")
    print(f" TRAINING PIPELINE FOR: 20{year} ")
    print(f"{'='*55}")

    # 1. Load Data
    file_path = os.path.join(DATA_DIR, file_name)
    df = pd.read_csv(file_path)

    # 2. Handle Sparsity: Drop rows where any of the 3 targets are missing
    df_clean = df.dropna(subset=TARGETS)
    print(f"Data shape after dropping missing targets: {df_clean.shape}")

    # 3. Isolate Features (X) and Targets (Y)
    Y = df_clean[TARGETS]
    X = df_clean.drop(columns=['CHESS', 'YEAR'] + TARGETS)

    # 4. Define the Optuna Objective
    def objective(trial):
        alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        n_estimators = trial.suggest_int('n_estimators', 10, 50)

        base_model = MultiOutputRegressor(
            ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
        )

        bagged_model = BaggingRegressor(
            estimator=base_model,
            n_estimators=n_estimators,
            random_state=42,
            n_jobs=-1 # Utilize all CPU cores
        )

        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', bagged_model)
        ])

        # Robust evaluation for small data
        cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
        scores = cross_val_score(pipeline, X, Y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)

        return scores.mean()

    # 5. Run Optimization with Progress Bar
    study = optuna.create_study(direction='maximize')

    with tqdm(total=100, desc=f"Tuning 20{year} Model", leave=True) as pbar:
        def callback(study, trial):
            pbar.update(1)
        study.optimize(objective, n_trials=100, callbacks=[callback])

    best_params = study.best_params
    print(f"\n[Optuna Phase Complete]")
    print(f"Best Hyperparameters found: {best_params}")
    print(f"Cross-Validated MSE:        {-study.best_value:.4f}")

    # 6. Train Final Model on the Complete Cleaned Dataset
    final_base = MultiOutputRegressor(
        ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42)
    )
    final_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', BaggingRegressor(
            estimator=final_base,
            n_estimators=best_params['n_estimators'],
            random_state=42,
            n_jobs=-1
        ))
    ])

    final_model.fit(X, Y)

    # 7. Evaluate Model (Full Training Set)
    Y_pred = final_model.predict(X)
    mse = mean_squared_error(Y, Y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(Y, Y_pred)
    r2 = r2_score(Y, Y_pred)

    print("\n[Final Evaluation Metrics]")
    print(f"MSE:  {mse:.4f} (Closer to 0 is better)")
    print(f"RMSE: {rmse:.4f} (Average error in ideology scale units)")
    print(f"MAE:  {mae:.4f} (Absolute average error)")
    print(f"R²:   {r2:.4f} (Closer to 1 is better)")

    # 8. Save the Model
    save_path = os.path.join(MODEL_DIR, f'ideology_model_20{year}.pkl')
    joblib.dump(final_model, save_path)
    print(f"\n✓ Model successfully saved to: {save_path}")

print("\nAll models trained and exported successfully.")